In [ ]:
# Common imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve
sns.set(style='whitegrid')


## Chargement et préparation des données

In [ ]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape)


## Decision Tree (CART)
Train an initial DecisionTreeClassifier, visualize it and study the effect of max_depth.


In [ ]:
# Train DecisionTreeClassifier (default params)
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)
print('Test accuracy (default DT):', clf.score(X_test, y_test))


In [ ]:
# Visualize tree (first 3 levels)
plt.figure(figsize=(12,6))
plot_tree(clf, max_depth=3, feature_names=data.feature_names, class_names=data.target_names, filled=True)
plt.show()


In [ ]:
# TODO: Study effect of max_depth

#    - Train multiple Decision Trees with varying max_depth values.
#    - Record training and test accuracies.
#    - Visualize how model complexity affects overfitting/underfitting.
# Steps:
#    1- Create a list of depths (e.g., 1 → 15)
#    2- Train a DecisionTreeClassifier for each depth
#    3- Compute accuracy on both training and test sets
#    4- Plot curves of accuracy vs depth
#    5- Observe where overfitting starts

# Hints:
# - Use two lists: train_acc and test_acc
# - Use plt.plot() to visualize
# - Use random_state=42 for reproducibility
depths = range(1, 16)
train_acc = []
test_acc = []
for d in depths:
     model = DecisionTreeClassifier(max_depth=d, random_state=42)
     model.fit(X_train, y_train)
     train_acc.append(model.score(X_train, y_train))
     test_acc.append(model.score(X_test, y_test))

plt.plot(depths, train_acc, label='Train')
plt.plot(depths, test_acc, label='Test')
plt.xlabel('Max depth')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Effect of max_depth on accuracy')
plt.show()


### Question 1
Quelle profondeur minimise le sur-apprentissage et pourquoi ?


## Random Forest
Train RandomForest. Compare performance and stability.


In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print('Random Forest test accuracy:', rf.score(X_test, y_test))

# Feature importances
importances = pd.Series(rf.feature_importances_, index=data.feature_names).sort_values(ascending=False)
print(importances.head(10))


## Boosting (AdaBoost & Gradient Boosting)
Train AdaBoost and GradientBoosting and compare to previous methods.


In [ ]:
# AdaBoost from scratch —
# TODO: Implement AdaBoost from scratch (binary classification)
# ---------------------------------------------
# Steps reminder:
# 1. Initialize sample weights equally.
# 2. For each iteration t:
#     a. Train a weak classifier (Decision Stump: DecisionTreeClassifier(max_depth=1))
#     b. Compute weighted error.
#     c. Compute alpha_t = 0.5 * log((1 - error) / error)
#     d. Update sample weights.
#     e. Normalize weights.
# 3. Final prediction = sign(sum(alpha_t * h_t(x)))



In [ ]:
# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
print('Gradient Boosting test accuracy:', gb.score(X_test, y_test))


Gradient Boosting test accuracy: 0.9473684210526315


### Question 2
Expliquez pourquoi le boosting peut réduire le biais mais être sensible au bruit (sur-apprentissage).


## Comparison and visualizations
Build a summary table and ROC curves for the models.


In [ ]:
# Compare models and build summary table
models = {'DecisionTree': clf, 'RandomForest': rf, 'GradBoost': gb}
results = []
for name, m in models.items():
     y_pred = m.predict(X_test)
     acc = accuracy_score(y_test, y_pred)
     results.append({'Model': name, 'Accuracy': acc})
# TODO: Add AdaBoost from scratch results

res_df = pd.DataFrame(results)
res_df


### Question 3
Interprétez les résultats : pour quel cas d'usage (faible bruit, beaucoup de bruit, besoin d'interprétabilité) recommanderiez-vous chaque méthode ?


## Stacking
Implémentez un stacking simple : combinez 3 modèles (ex: LogisticRegression, RandomForest, GradientBoosting) via un meta-estimator (LogisticRegression).


In [ ]:
# TODO : Stacking
# Objective:
#    - Combine several classifiers (heterogeneous) into one meta-model.
#    - Compare its performance to single models.
#
# Predefined base models:
#    1- Logistic Regression
#    2- Random Forest
#    3- Gradient Boosting
# Meta-model: Logistic Regression
#
# Steps:
#    1. Define the base learners (estimators list)
#    2. Define the final estimator (meta-model)
#    3. Train a StackingClassifier
#    4. Evaluate its test accuracy
#    5. Compare to previous models

# Hints:
# - Use StackingClassifier from sklearn.ensemble
# - You can use 5-fold cross-validation (cv=5)
# - Compare accuracy with RF, GB, AdaBoost
